In [ ]:
import polars as pl
import numpy as np
import matplotlib as plt
import seaborn as sns
import datetime as dt 

ENDERECO_DADOS = r'C:/Users/priscilla.goncalves/Documents/analise-dados/Analise_modulo2/dados_bronze/'


In [10]:
try:
    print(f'Lendo arquivo parquet')

    hora_início =dt.datetime.now()

# # cria um plano de execução sobre o futuro df
    df_bf_exec_plan = pl.scan_parquet(ENDERECO_DADOS + 'df_bf.parquet')
    
    df_cidades = pl.scan_csv(ENDERECO_DADOS + '14.CodigosMunic.csv', separator=';')
    
    df_lazy = (df_bf_exec_plan.join(df_cidades, left_on='CÓDIGO MUNICÍPIO SIAFI', right_on='COD_SIAFI', how='left').select(
        [pl.col('VALOR PARCELA'), pl.col('UF'), pl.col('MÊS COMPETÊNCIA'), pl.col('NOME')]).with_columns(
        pl.col('MÊS COMPETÊNCIA').cast(pl.Utf8).str.strptime(pl.Date, format='%Y%m')))

    
# # vira um dataframe quando eu coleto
    # df_bf = df_bf_exec_plan.collect()
    # df_bf = pl.read_parquet(ENDERECO_DADOS + 'df_bf.parquet')
    hora_fim = dt.datetime.now()
    # print(df_bf.columns)
    # print(df_bf.head())
    print(f'Tempo de execução: {hora_fim - hora_início}')

except Exception as e:
    print(f'Erro ao carregar dados: {e}')

Lendo arquivo parquet
Tempo de execução: 0:00:00


In [ ]:
# analise exploratoria basica
coluna_alvo = 'VALOR PARCELA'

df_analises = df_lazy.select(pl.col(coluna_alvo).mean().alias('Media'), 
                             pl.col(coluna_alvo).median().alias('Mediana/Q2'),
                             pl.col(coluna_alvo).std().alias('Desvio_padrão'), 
                             pl.col(coluna_alvo).skew().alias('Assimetria'),
                             pl.col(coluna_alvo).kurtosis().alias('Curtose'),
                             (pl.col(coluna_alvo).mean() - pl.col(coluna_alvo).median()).alias('delta_media_mediana'), 
                             pl.col(coluna_alvo).quantile(0.25).alias('Q1'),
                             pl.col(coluna_alvo).quantile(0.75).alias('Q3')).collect()                       


In [16]:
try:

    print(f'Preparando dados para a visualização')

    df = df_lazy.collect()

    valores = np.array(df['VALOR PARCELA'])

    q1 = df_analises['Q1'][0]
    q3 = df_analises['Q3'][0]
    iqr = q3 - q1

    lim_sup = q3 + 1.5 * iqr
    lim_inf = q1 - 1.5 * iqr
    outliers_sup = valores > lim_sup
    outliers_inf = valores < lim_inf

    total_outliers = len(outliers_inf) + len(outliers_sup)

    porcentagem_outliers = (total_outliers/len(valores)) * 100

    print(valores)
    print(lim_inf)
    print (total_outliers)


except Exception as e:
    print (f' Erro na preparação dos dados: {e}')

Preparando dados para a visualização
[650. 650. 650. ... 600. 600. 650.]
375.0
122811368
